# End of week 1 exercise

To demonstrate your familiarity with OpenAI API, and also Ollama, build a tool that takes a technical question,  
and responds with an explanation. This is a tool that you will be able to use yourself during the course!

In [8]:
# imports
import os
from dotenv import load_dotenv
from enum import Enum
from dataclasses import dataclass
from openai import OpenAI
from IPython.display import Markdown, display, update_display


In [14]:
# Initialize

load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')
ollama_api_key = "ollama_api_key"

if openai_api_key and openai_api_key.startswith('sk-proj-') and len(openai_api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")

# constants
class ModelProvider(str, Enum):
    OPENAI = 'openai'
    OLLAMA = 'ollama'

@dataclass
class ModelConfig:
    model_name: str
    api_key: str
    url: str | None = None

MODEL_CONFIGS: dict[ModelProvider, ModelConfig] = {
    ModelProvider.OPENAI: ModelConfig(model_name='gpt-4o-mini', api_key=openai_api_key, url=None),
    ModelProvider.OLLAMA: ModelConfig(model_name='llama3.2', url='http://localhost:11434/v1', api_key=ollama_api_key),
}

API key looks good so far


In [18]:
# set up environment
system_prompt: str = """
You are expert AI Engineer, and has worked in Software Engineering for more than 20 years. 
You have vast technical knowledge around different technologies and tools. and you are so good and expert in LLM Engineering
that you got awarded as the best AI Engineer in the world.
You are so good in LLM Engineering that you can answer any question related to the LLM Engineering course with friendly and helpful manner.
and also you can answer the question with detailed explanation with examples.
Your answer is so good that it can be understand by any software engineer who has started to work in the field.
"""

def question_prompt(question: str) -> str:
    return f"""
    Here is the question:
    {question}
    """

def display_content(content: str, display_id: str | None = None) -> str:
    
    if not display_id:
        display_handle = display(Markdown(""), display_id=True)
        display_id = display_handle.display_id
    
    update_display(obj=Markdown(content), display_id=display_id)
    return display_id

def get_answer_to_question(question: str, model_provider: ModelProvider, stream: bool = True) -> str:
    model_config: ModelConfig = MODEL_CONFIGS[model_provider]
    model = OpenAI(api_key=model_config.api_key, base_url=model_config.url)

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": question_prompt(question)}
    ]

    response = model.chat.completions.create(
        model=model_config.model_name,
        messages=messages,
        stream=stream
    )

    content: str = ""
    if not stream:
        content = response.choices[0].message.content
        display_content(content=content)
    else:
        display_id: str | None = None
        for chunk in response:
            content += chunk.choices[0].delta.content or ''
            display_id = display_content(content=content, display_id=display_id)

    return content

In [19]:
# here is the question; type over this to ask something new

question = """
Please explain what this code does and why:
yield from {book.get("author") for book in books if book.get("author")}
"""

In [22]:
# Get gpt-4o-mini to answer, with streaming
result = get_answer_to_question(question=question, model_provider=ModelProvider.OPENAI, stream=True)

Certainly! Let's break down the code snippet you've provided:

```python
yield from {book.get("author") for book in books if book.get("author")}
```

### Explanation

1. **Context of the Code**: 
   - This line of code is typically part of a generator function in Python. The context involves yielding values, which means that it can be used in a function that produces an iterator. The `yield` keyword allows the function to pause and return a value to the caller, but it retains the function's state for when it resumes.

2. **Set Comprehension**:
   - The code uses a set comprehension: `{... for book in books if book.get("author")}`.
   - A set comprehension is a concise way to create a set (a collection of unique elements) in Python. It follows the format `{expression for item in iterable if condition}`.

3. **Iterating Over `books`**:
   - Here, `books` is assumed to be an iterable collection (like a list) of dictionaries, where each dictionary represents a book.
   - The `get` method accesses the value associated with the key `"author"` in each `book` dictionary. This method returns `None` if the key does not exist.

4. **Conditional Filtering**:
   - The condition `if book.get("author")` checks if the `"author"` exists and has a truthy value (i.e., it's not `None`, empty string, or some equivalent of false). This filters out any books that do not have an author specified.

5. **Creating the Set**:
   - The comprehension collects all unique authors from the `books` collection. If multiple books have the same author, that author will only appear once in the resulting set due to the nature of sets requiring all elements to be unique.

6. **Yielding the Results**:
   - The `yield from` statement allows the generator to yield all items from the generated set one-by-one. This means that when the generator is iterated over, it will return each unique author from the set.

### Example

Let's consider an example to illustrate how this works:

```python
books = [
    {"title": "Book One", "author": "Alice"},
    {"title": "Book Two", "author": "Bob"},
    {"title": "Book Three", "author": ""},
    {"title": "Book Four", "author": "Alice"},
    {"title": "Book Five"},
]

def unique_authors(books):
    yield from {book.get("author") for book in books if book.get("author")}

for author in unique_authors(books):
    print(author)
```

### Output

```
Alice
Bob
```

### Summary

- The line of code collects unique authors from a list of book dictionaries by checking for their existence and yielding them one by one.
- It elegantly combines set comprehension and generator functionality to make the code concise and readable.
- This method is efficient in ensuring only unique and existing authors are returned from the input collection. 

Don’t hesitate to ask if you have any more questions or need further clarification!

In [23]:
# Get Llama 3.2 to answer
result = get_answer_to_question(question=question, model_provider=ModelProvider.OLLAMA, stream=True)

This is a lovely piece of code.

**Code Breakdown**

Let's break it down:

`yield from`

In Python, `yield` is used to define generators. A generator is a special type of function that returns an iterator, which produces values one at a time.

`{book.get("author") for book in books if book.get("author")}`

This is a data structure called a **dictionary comprehension** or a **set comprehension**. It's equivalent to the following nested `for` loop:

```python
selected_authors = set()
for book in books:
    author = book.get("author")
    if author:
        selected_authors.add(author)
```

When we use `yield from`, it "yields" all values from the inner expression, one at a time. In this case, the inner expression is the dictionary comprehension.

**What's happening here?**

The code uses a technique called **yielding from an iterator**, which allows us to lazily iterate over a subset of values from another iterable (in this case, `books`). We're filtering out books without an `"author"` key using the `if book.get("author"):` condition.

Here's what's happening:

1. For each book in `books`, we check if it has an `"author"` key.
2. If the book has an `"author"` key, we add its value to a dictionary comprehension (which creates an iterator over possible authors).
3. We use `yield from` to yield one author at a time from this iterator.

**When would you use this code?**

This kind of code is often used when:

* You have a large dataset and want to process it in chunks, rather than loading the entire thing into memory.
* You need to filter out values based on some condition.
* You want to write concise and efficient code that's still easy to understand.

**Example Use Case**

Let's say we're working with a bookstore system (`books` is an iterable of books), and we want to print only the authors from books written by a specific author (e.g., Shakespeare).

```python
shakespeare_books = [book for book in books if book.get("author") == "Shakespeare"]
yield from {book.get("author") for book in shakespeare_books}
```

In this example, we'd first filter out all non-Shakespeare books and then yield only the authors of those books.

By using `yield from` with a dictionary comprehension, we've created a concise and efficient way to generate authors without having to use a separate `for` loop or other iteration constructs.